# 03 — Forecasting & predictive analytics (demo)

Model outputs, backtest grids, drift — **synthetic** metrics for pipeline testing.

See `features/03-forecasting-and-predictive-analytics.md`.


In [ ]:
import os
import sys
from pathlib import Path

# Uploaded demo modules (Databricks). `dbfs:/tmp/...` is readable on shared UC clusters; FileStore path is a fallback.
sys.path.insert(0, "/dbfs/tmp/energy-trading-forecast-demo")
sys.path.insert(0, "/dbfs/FileStore/energy-trading-forecasting-demo/demo_data")
_env = os.environ.get("DEMO_DATA_PATH", "").strip()
if _env:
    sys.path.insert(0, _env)

CWD = Path.cwd()
for dd in (
    CWD / "modules" / "forecasting" / "demo_data",
    CWD / "demo_data",
    CWD.parent / "demo_data",
):
    if (dd / "notebook_helpers.py").is_file():
        sys.path.insert(0, str(dd))
        break

import notebook_helpers as nh
nh.ensure_demo_data_on_path()
# Matches modules/forecasting/README.md → unity_catalog.schemas (override with DEMO_UC_* env)
print(
    "Unity Catalog Delta target:",
    nh.catalog_schema(),
    "— example:",
    nh.full_table_name("demo_prices_spot_hourly"),
)

import synthetic_generators as sg

spark = nh.get_spark()


In [ ]:

nh.write_demo_table(
    "demo_forecasts_load_gen",
    sg.forecasts_load_gen_rows(),
    ("ts", "zone", "model_id", "load_fcast_mw", "wind_fcast_mw", "solar_fcast_mw", "residual_mw"),
    spark=spark,
)
nh.write_demo_table(
    "demo_forecasts_market_prices",
    sg.forecasts_market_prices_rows(),
    ("ts", "zone", "product", "mid_eur_mwh", "q10_eur_mwh", "q90_eur_mwh", "imbalance_risk_index"),
    spark=spark,
)
nh.write_demo_table(
    "demo_carbon_spark_daily",
    sg.carbon_spark_rows(),
    ("ts", "zone", "eua_eur_t", "goo_eur_mwh", "ttf_eur_mwh", "clean_spark_proxy"),
    spark=spark,
)
nh.write_demo_table(
    "demo_backtest_metrics",
    sg.backtest_metrics_rows(),
    ("run_id", "model_id", "product", "zone", "horizon", "mae", "rmse", "mape", "skill", "pinball"),
    spark=spark,
)
nh.write_demo_table(
    "demo_drift_metrics",
    sg.drift_metrics_rows(),
    ("as_of_date", "model_id", "feature_group", "zone", "psi"),
    spark=spark,
)
nh.write_demo_table(
    "demo_regime_labels",
    sg.regime_labels_rows(),
    ("regime_id", "driver", "effect", "month_tag"),
    spark=spark,
)
print("Forecasting demo tables written.")
